# Calibration, Thresholds, and Decision Diagnostics Lab


## Purpose

This lab explores calibration and threshold choice.

A model may rank cases well but still produce poorly calibrated probabilities.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 2000
score = rng.beta(2.2, 3.0, size=n)
target = rng.binomial(1, score)

frame = pd.DataFrame({"score": score, "target": target})

frame["confidence_bin"] = pd.cut(
    frame["score"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)

calibration = (
    frame
    .groupby("confidence_bin", observed=True)
    .agg(
        n=("target", "size"),
        mean_confidence=("score", "mean"),
        empirical_rate=("target", "mean"),
    )
    .reset_index()
)

calibration["absolute_gap"] = (
    calibration["mean_confidence"] - calibration["empirical_rate"]
).abs()

ece = (
    calibration["n"] / calibration["n"].sum() * calibration["absolute_gap"]
).sum()

calibration, ece

In [ ]:
threshold_rows = []

for threshold in np.linspace(0.1, 0.9, 9):
    prediction = (frame["score"] >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "selection_rate": prediction.mean(),
        "positive_predictive_value": frame.loc[prediction == 1, "target"].mean(),
    })

pd.DataFrame(threshold_rows)

## Governance Question

A threshold is not just a technical parameter. It often encodes an operational policy choice.